# Animal Eye Metrology — Demo

直接呼叫專案 Python 程式，展示 **動物實例分割 → 眼點定位 → 眼部分割 → 遮罩重心距離**。
沿用既有 YOLO26s-seg、HRNet-W32／AP-10K、SAM 2.1 Tiny 與核准設定。

| 模式 | 用途 | 是否執行模型 |
| --- | --- | --- |
| `replay`（預設） | 重播明確指定的本機 run；缺少 run 時清楚退回已提交 CSV | 否 |
| `live` | 對指定的本機 JPEG／PNG 呼叫完整推論 | 是 |
| `csv` | 只有儲存庫也能展示既有距離與公式驗算 | 否 |

**展示界線：** 所有輸出是 `candidate / unverified` 像素量測，並非瞳孔中心或實體長度。
解剖左右與可見性尚未獨立驗證；無人工參考時不產生 IoU、Dice、距離誤差或信賴區間。
預設重播範例來自 COCO 2017 validation 開發圖 `125405` 的歷史輸出，不是獨立測試集或本次推論成績。

先依 [使用說明](../docs/notebook_demo.md) 建立展示環境，再由上到下執行。
此 Notebook 不會下載資料或模型。

## 1. Environment and Demo Settings

從專案根目錄或 `notebooks/` 啟動；程式會向上尋找專案。
修改下方設定即可切換模式、指定既有 run 或換成自己的本機圖片。
`COMPARE_R050` 僅控制現場推論，重播沿用原 run 條件；表格保留所有條件與失敗列。

In [ ]:
import csv
import html
import importlib.metadata
import json
import math
import os
import sys
from pathlib import Path
from typing import Any

import numpy as np
from IPython.display import HTML, Markdown, display
from PIL import Image

roots = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
ROOT = next(
    (path for path in roots if (path / "src/animal_eye_metrology").is_dir()),
    None,
)
if ROOT is None:
    raise RuntimeError("請從 animal-eye-metrology 專案根目錄或 notebooks/ 啟動。")
sys.path.insert(0, str(ROOT / "src"))

In [ ]:
from animal_eye_metrology import __version__
from animal_eye_metrology.inference_artifacts import animal_segmentation_views
from animal_eye_metrology.inference_pipeline import (
    load_config,
    load_result,
    preflight,
    run_inference,
)
from animal_eye_metrology.metrology import euclidean_distance, mask_centroid
from animal_eye_metrology.verification import sha256_file

MODE = "replay"  # "replay", "live", or "csv"
REPLAY_RUN = ROOT / "outputs/inference/20261108T075732Z_92c42f74bb73"
IMAGE_PATH = ROOT / "data/coco/val2017/000000125405.jpg"
CONFIG_PATH = Path(os.environ.get("AEM_INFERENCE_CONFIG", "configs/inference.runtime.toml"))
if not CONFIG_PATH.is_absolute():
    CONFIG_PATH = ROOT / CONFIG_PATH
COMPARE_R050 = False
DISPLAY_CONDITION = "A_r100"
MAX_EYE_ZOOMS = 8
CSV_PATH = ROOT / "examples/measurements.csv"
CSV_SHA256 = "4a8ea8409a1d63bc5c3efd35b446c5ce17bcbf5f860ef686f5d1132a8b41bf11"

print(f"Project: {ROOT}")
print(f"Python: {sys.version.split()[0]} | kernel: {sys.executable}")
print(f"Project version: {__version__} | requested mode: {MODE}")
print(f"Live inference configuration: {CONFIG_PATH}")

In [ ]:
def show_table(rows: list[dict[str, Any]], columns: list[str] | None = None) -> None:
    """Display full-precision values; preserve missing values and escape HTML."""
    if not rows:
        display(Markdown("此步驟沒有記錄；請查看階段狀態與失敗原因。"))
        return
    columns = columns or list(dict.fromkeys(key for row in rows for key in row))

    def value_text(value: Any) -> str:
        if value is None:
            return "—"
        if isinstance(value, list | dict):
            return json.dumps(value, ensure_ascii=False, allow_nan=False)
        return str(value)

    header = "".join(f"<th>{html.escape(key)}</th>" for key in columns)
    body = "".join(
        "<tr>"
        + "".join(f"<td>{html.escape(value_text(row.get(key)))}</td>" for key in columns)
        + "</tr>"
        for row in rows
    )
    display(HTML(f'<div style="overflow-x:auto"><table><tr>{header}</tr>{body}</table></div>'))


def run_asset(relative: str | None) -> Path | None:
    """Resolve existing files inside the selected run; never write or fetch assets."""
    if run_dir is None or not relative:
        return None
    root = run_dir.resolve()
    path = (root / relative).resolve()
    if Path(relative).is_absolute() or not path.is_relative_to(root):
        raise ValueError(f"Artifact is outside the selected run: {relative}")
    return path if path.is_file() else None


def show_image(path: Path | None, caption: str) -> None:
    display(Markdown(f"**{caption}**"))
    if path is None:
        display(Markdown("本機缺少此影像產物；不以合成圖替代。"))
        return
    with Image.open(path) as image:
        display(image.copy())


show_table(
    [
        {"package": name, "version": importlib.metadata.version(name)}
        for name in ("numpy", "Pillow", "ipykernel", "jupyterlab")
    ]
)

## 2. Load a Recorded Run or Execute the Pipeline

`live` 先使用既有 `preflight()` 檢查來源、直譯器與權重雜湊，通過後才執行。
Notebook 使用輕量環境；模型仍由設定檔指定的 app／pose 子程序隔離執行。
若現場執行失敗，保留並展示原始狀態，不改用舊結果冒充成功。
CPU 執行時間依機器與圖片而異；重新執行此儲存格會建立另一個 run。

In [ ]:
if MODE not in {"replay", "live", "csv"}:
    raise ValueError("MODE must be 'replay', 'live', or 'csv'.")

run_dir = None
result = None
actual_mode = MODE

if MODE == "live":
    cfg = load_config(CONFIG_PATH)
    checks = preflight(cfg)
    show_table(checks, ["component", "status", "detail"])
    if any(row["status"] != "ready" for row in checks):
        raise RuntimeError("現場推論環境未就緒；請依 setup 文件準備或明確選擇既有設定。")
    if not IMAGE_PATH.is_file():
        raise FileNotFoundError(f"請將 IMAGE_PATH 指向已存在的本機 JPEG／PNG：{IMAGE_PATH}")
    conditions = ("A_r100", "A_r050") if COMPARE_R050 else ("A_r100",)
    run_dir = run_inference(
        image_bytes=IMAGE_PATH.read_bytes(),
        filename=IMAGE_PATH.name,
        config_path=CONFIG_PATH,
        conditions=conditions,
        progress=lambda stage, status: print(f"{stage}: {status}", flush=True),
    )
    result = load_result(run_dir)
elif MODE == "replay":
    if (REPLAY_RUN / "result.json").is_file():
        run_dir = REPLAY_RUN.resolve()
        result = load_result(run_dir)
    else:
        actual_mode = "csv"
        display(Markdown("**指定的 run 不在本機，已退回既有 CSV；沒有執行模型或產生圖像。**"))

if result is None:
    if sha256_file(CSV_PATH) != CSV_SHA256:
        raise ValueError("Recorded CSV SHA-256 mismatch; inspect examples/README.md.")
    with CSV_PATH.open(encoding="utf-8-sig", newline="") as handle:
        measurements = list(csv.DictReader(handle))
    numeric_fields = ("cx_a_px", "cy_a_px", "cx_b_px", "cy_b_px", "distance_px")
    for row in measurements:
        for field in numeric_fields:
            row[field] = float(row[field]) if row[field] else None
    source_label = "既有 CSV；不是現場推論，沒有原圖或遮罩"
else:
    measurements = result.get("measurements", [])
    source_label = "本次現場推論" if actual_mode == "live" else "指定的既有 run；不是現場推論"

display(Markdown(f"**資料來源：{source_label}**"))
print(f"Run directory: {run_dir}")
print(f"Measurement records: {len(measurements)}")
if result is not None:
    show_table(
        [
            {
                "run_id": result.get("run_id"),
                "status": result.get("status"),
                "created_at_utc": result.get("created_at_utc"),
                "elapsed_seconds_at_run": result.get("elapsed_seconds"),
                "conditions": result.get("conditions"),
                "fatal_error": result.get("fatal_error"),
            }
        ]
    )
    show_table(
        [{"stage": stage, **values} for stage, values in result.get("stages", {}).items()],
        ["stage", "status", "reason", "seconds"],
    )

## 3. Original Image and Animal Instance Segmentation

使用保存的正規化原圖及實例遮罩；不把偵測框當作分割結果。
每個動物使用不同顯示顏色。原圖未縮放，所有量測維持原圖座標。
表格包含接受與未接受的候選，並分開記錄動物遮罩的狀態。

In [ ]:
if result is None:
    display(Markdown("CSV 模式只有既有數值；完整影像 Demo 需要本機 run 或現場推論。"))
else:
    show_image(run_asset(result["input"]["image_path"]), "Original normalized image")
    animals = result.get("animals", [])
    show_table(
        animals,
        [
            "prediction_id",
            "category",
            "score",
            "accepted",
            "supported",
            "status",
            "mask_status",
            "mask_area_px",
            "failure_reason",
            "mask_failure_reason",
            "experimental",
            "pose_domain",
        ],
    )
    needed_masks = [row.get("mask_path") for row in animals if row.get("accepted")]
    if run_asset(result["input"]["image_path"]) and all(
        not relative or run_asset(relative) is not None for relative in needed_masks
    ):
        views = animal_segmentation_views(run_dir, result, show_boxes=False)
        for name, image in views.items():
            display(Markdown(f"**Animal segmentation: {name}**"))
            display(image)
    else:
        display(Markdown("本機缺少原圖或已登記的實例遮罩，略過輪廓顯示；狀態表仍完整保留。"))

## 4. Eye Localization, Segmentation and Review

眼點定位、眼部分割及量測資格分開展示。
`left/right` 是模型的解剖通道，不依畫面 x 座標重新排序；沒有獨立左右與可見性驗證。
疊圖：**青色 L、紅色 R、十字為 HRNet 提示點、圓圈為 SAM 遮罩重心**；
白線為同隻雙眼、紫線為跨動物右眼候選距離。

下方放大圖使用記錄的原圖 ROI，僅供目視檢查，不會改變遮罩或重新計算模型輸出。
模型品質分數不是已校準的正確率；鳥類輸出需保留域外試驗標記。

In [ ]:
if result is not None:
    show_table(
        result.get("eyes", []),
        [
            "prediction_id",
            "category",
            "condition_id",
            "eye_side",
            "localization_status",
            "segmentation_status",
            "point_score",
            "mask_area_px",
            "centroid_x_px",
            "centroid_y_px",
            "visibility_status",
            "side_assignment_status",
            "measurement_qualification",
            "experimental",
            "pose_domain",
            "failure_reason",
        ],
    )
    for condition in result.get("conditions", []):
        relative = result.get("artifacts", {}).get(f"{condition}_overlay")
        show_image(run_asset(relative), f"{condition}: saved eye masks and candidate distances")
else:
    display(Markdown("CSV 模式沒有逐眼定位、遮罩或疊圖，不以距離表推斷分割正確。"))

In [ ]:
if result is not None:
    selected_eyes = [
        row for row in result.get("eyes", []) if row["condition_id"] == DISPLAY_CONDITION
    ]
    image_path = run_asset(result["input"]["image_path"])
    if image_path is not None:
        with Image.open(image_path) as original:
            for eye in selected_eyes[:MAX_EYE_ZOOMS]:
                display(
                    Markdown(
                        f"**{eye['prediction_id']} / {eye['eye_side']} / {DISPLAY_CONDITION}**"
                    )
                )
                roi = eye.get("roi_xyxy")
                mask_path = run_asset(eye.get("mask_path"))
                if roi is None or mask_path is None:
                    display(Markdown(f"無可用 ROI／遮罩：{eye.get('failure_reason', '')}"))
                    continue
                crop = original.convert("RGB").crop(tuple(roi))
                with Image.open(mask_path) as mask:
                    crop_mask = mask.convert("L").crop(tuple(roi))
                color = (35, 215, 245) if eye["eye_side"] == "left" else (245, 85, 110)
                tint = Image.blend(crop, Image.new("RGB", crop.size, color), 0.65)
                overlay = Image.composite(tint, crop, crop_mask)
                scale = min(6, max(1, 320 // max(crop.size)))
                size = (crop.width * scale, crop.height * scale)
                display(crop.resize(size, Image.Resampling.NEAREST))
                display(overlay.resize(size, Image.Resampling.NEAREST))
        print(f"Shown {min(len(selected_eyes), MAX_EYE_ZOOMS)}/{len(selected_eyes)} eye ROIs.")
        print("Zooms are display-only; the eye table above retains every record.")

## 5. Candidate Distance Measurements

| 類型 | 定義 |
| --- | --- |
| `binocular` | 同隻動物左眼與右眼遮罩重心 |
| `right_to_right` | 同圖兩隻不同動物的右眼遮罩重心 |

二值遮罩前景的像素中心平均為 $(c_x,c_y)=(\operatorname{mean}(column)+0.5,\operatorname{mean}(row)+0.5)$；
距離為 $d=\sqrt{(c_{x,a}-c_{x,b})^2+(c_{y,a}-c_{y,b})^2}$，單位 **px**。

所有距離、條件、狀態、資格與缺失原因都保留。端點缺失時距離維持 `None`／空白，不能補零。
A_r100 與 A_r050 是同張圖的不同條件，不是獨立樣本；差值不等於真實量測誤差或不確定度。

In [ ]:
show_table(
    measurements,
    [
        "condition_id",
        "measurement_type",
        "prediction_id_a",
        "eye_side_a",
        "prediction_id_b",
        "eye_side_b",
        "cx_a_px",
        "cy_a_px",
        "cx_b_px",
        "cy_b_px",
        "distance_px",
        "units",
        "status",
        "qualification",
        "experimental",
        "pose_domain_a",
        "pose_domain_b",
        "reason",
    ],
)

## 6. Verify Geometry with the Project Functions

以專案的 `euclidean_distance()` 重新驗算每列端點；有原始遮罩時，再用 `mask_centroid()`
核對保存的重心。這驗證**幾何公式與產物一致性**，不證明眼睛定位、分割、左右或距離準確。
缺少遮罩會標為未驗證，而不是視為通過。

In [ ]:
def endpoint(row: dict[str, Any], suffix: str) -> tuple[float, float] | None:
    x, y = row.get(f"cx_{suffix}_px"), row.get(f"cy_{suffix}_px")
    if x is None and y is None:
        return None
    if x is None or y is None:
        raise ValueError("Incomplete centroid coordinates in a measurement row.")
    return float(x), float(y)


verification = []
for row in measurements:
    recalculated = euclidean_distance(endpoint(row, "a"), endpoint(row, "b"))
    recorded = row.get("distance_px")
    matches = (
        recorded is None
        if recalculated is None
        else recorded is not None and math.isclose(recalculated, recorded, rel_tol=1e-12)
    )
    if not matches:
        raise ValueError(f"Saved distance disagrees with endpoints: {row}")
    verification.append(
        {
            "condition_id": row["condition_id"],
            "measurement_type": row["measurement_type"],
            "recorded_px": recorded,
            "recalculated_px": recalculated,
            "formula_consistent": matches,
        }
    )
show_table(verification)

centroid_checks = []
if result is not None:
    for eye in result.get("eyes", []):
        path = run_asset(eye.get("mask_path"))
        state = "not_checked: mask unavailable"
        if path is not None:
            with Image.open(path) as mask:
                center = mask_centroid(np.asarray(mask.convert("L")) > 0)
            stored = (eye.get("centroid_x_px"), eye.get("centroid_y_px"))
            matches = (
                stored == (None, None)
                if center is None
                else all(
                    value is not None and math.isclose(value, actual, rel_tol=1e-12)
                    for value, actual in zip(stored, center, strict=True)
                )
            )
            if not matches:
                raise ValueError(f"Saved centroid disagrees with binary mask: {eye}")
            state = "consistent" if center is not None else "consistent: empty mask, no centroid"
        centroid_checks.append(
            {
                "prediction_id": eye["prediction_id"],
                "eye_side": eye["eye_side"],
                "condition_id": eye["condition_id"],
                "mask_centroid_check": state,
            }
        )
show_table(centroid_checks) if result is not None else print("CSV-only: no mask-centroid check.")
print("Geometry consistency only; no accuracy, uncertainty or eligibility validation.")

## 7. Provenance, Outputs and Limitations

重播時下表使用 **run 當時保存的設定**，不是目前設定檔；套件與原始碼身份以原 run manifest 為準。
現場推論沿用原程式保存：輸入與權重雜湊、設定、模型執行命令、階段日誌、CSV／JSON／PNG。
Notebook kernel 的套件版本與模型子程序版本分開記錄。

In [ ]:
if result is None:
    show_table(
        [
            {
                "source": str(CSV_PATH.relative_to(ROOT)),
                "sha256": sha256_file(CSV_PATH),
                "integrity": "matches recorded example",
                "provenance": "examples/README.md",
                "dataset": "COCO 2017 validation, development image 125405",
            }
        ]
    )
else:
    saved_cfg = result.get("config", {})
    fields = [
        "device",
        "seed",
        "threads",
        "supported_categories",
        "animal_score_threshold",
        "image_size",
        "detector_checkpoint_sha256",
        "pose_checkpoint_sha256",
        "sam_checkpoint_sha256",
        "mmpose_commit",
        "source",
    ]
    show_table([{"setting_at_run": key, "value": saved_cfg.get(key)} for key in fields])
    show_table([result.get("input", {})])
    records = []
    for filename in ("result.json", "run_manifest.json", "config.toml"):
        path = run_asset(filename)
        records.append(
            {
                "record": filename,
                "sha256_now": sha256_file(path) if path else None,
                "available": path is not None,
            }
        )
    show_table(records)
    manifest_path = run_asset("run_manifest.json")
    if manifest_path is not None:
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        integrity_checks = []
        for relative, expected in manifest.get("files_sha256", {}).items():
            path = run_asset(relative)
            current = sha256_file(path) if path is not None else None
            state = (
                "missing" if current is None else "matches" if current == expected else "MISMATCH"
            )
            integrity_checks.append({"file": relative, "manifest_check": state})
        show_table(integrity_checks)
        if any(row["manifest_check"] == "MISMATCH" for row in integrity_checks):
            raise ValueError("Run artifact SHA-256 mismatch; inspect the integrity table.")
    else:
        display(Markdown("缺少 run manifest：只列出目前雜湊，未驗證產物完整性。"))
    show_table(
        [
            {
                "artifact_key": key,
                "relative_path": relative,
                "available": run_asset(relative) is not None,
            }
            for key, relative in result.get("artifacts", {}).items()
        ]
    )
    show_table(result.get("execution", []))
    display(Markdown("**原 run 限制（完整保留）：**"))
    for limitation in result.get("limitations", []):
        print(f"- {limitation}")
    print(f"Outputs remain in: {run_dir}")

print(f"Actual demo mode: {actual_mode}; source: {source_label}")
print("No GT was loaded; no IoU/Dice, measurement error or confidence interval was estimated.")

## Demo Talking Points

1. **系統分工：** 動物實例、眼點與眼遮罩是不同任務；Notebook 直接重用同一套 Python 流程。
2. **量測定義：** 展示同隻雙眼與跨動物右眼的遮罩重心像素距離，說明左右通道與資格尚未驗證。
3. **工程證據：** 展示逐階段狀態、缺失原因、CSV／JSON、來源雜湊及固定設定。
4. **研究界線：** 幾何可計算與模型成功不代表量測準確；後續需獨立人工參考與核准評估。

重新展示可使用 **Restart Kernel and Run All**；`live` 會再建立一次新 run，`replay/csv` 只讀取既有資料。
提交前清除輸出，避免將圖片、個人路徑或大型結果內嵌進 Git。